In [ ]:
%load_ext autoreload
%autoreload 2

In [1]:
from sc_exp_design.couplings import IndependentCoupling
from sc_exp_design.data import DataManager, ValidationDataLoader
from sc_exp_design.sym.dummy_data import get_dummy_adata


In [2]:
adata = get_dummy_adata()
adata

AnnData object with n_obs × n_vars = 850 × 200
    obs: 'is_control', 'target0', 'target1', 'target2', 'treatment0', 'treatment1'
    uns: 'treatment0_label', 'treatment0_group', 'treatment1_label', 'treatment1_group'
    obsm: 'states', 'treatment0_dose', 'treatment0_time', 'treatment1_dose', 'treatment1_time', 'treatment2_features', 'target3'

In [3]:
def split_adata(
    adata,
    obs_column,
    unique_value_ids,
):
    """"""
    import numpy as np

    if isinstance(unique_value_ids, int):
        unique_value_ids = [unique_value_ids, ]

    # retrieving unique values of column and performing sanity check
    unique_vals = adata.obs[obs_column].unique()
    for val_id in unique_value_ids:
        assert val_id < len(unique_vals)
    
    # store for ood indices and adatas
    odd_idxs_dict = {}
    odd_adatas_dict = {}

    # iterating over each unique value index to leave out
    for val_id in unique_value_ids:
        # retrieving corresponding value and computing mask
        ood_val = unique_vals[val_id]
        ood_idxs = adata.obs[obs_column].map(lambda e: e == ood_val).values

        # storing indices and sliced adata
        odd_idxs_dict[val_id] = ood_idxs
        odd_adatas_dict[val_id] = adata[ood_idxs].copy()

    # slicing the data
    is_ood = np.any(
        np.stack(list(odd_idxs_dict.values()), axis=0), axis=0
    )
    train_adata = adata[~is_ood].copy()
    return train_adata, odd_adatas_dict

train_adata, ood_adata = split_adata(adata, "treatment0", 1)
ood_adata = ood_adata[1]
train_adata, ood_adata

(AnnData object with n_obs × n_vars = 700 × 200
     obs: 'is_control', 'target0', 'target1', 'target2', 'treatment0', 'treatment1'
     uns: 'treatment0_label', 'treatment0_group', 'treatment1_label', 'treatment1_group'
     obsm: 'states', 'treatment0_dose', 'treatment0_time', 'treatment1_dose', 'treatment1_time', 'treatment2_features', 'target3',
 AnnData object with n_obs × n_vars = 150 × 200
     obs: 'is_control', 'target0', 'target1', 'target2', 'treatment0', 'treatment1'
     uns: 'treatment0_label', 'treatment0_group', 'treatment1_label', 'treatment1_group'
     obsm: 'states', 'treatment0_dose', 'treatment0_time', 'treatment1_dose', 'treatment1_time', 'treatment2_features', 'target3')

In [13]:
dm = DataManager(
    train_adata,
    sample_rep="states",
    control_key="is_control",
    perturbations=("treatment0", "treatment1"),
    perturbation_reps={"treatment0":("treatment0_label",),  "treatment1": ("treatment1_label",)},
    load_target_covariates=True,
    target_covariates={"target0":"label"}
)
dm = DataManager(
    train_adata,
    sample_rep="states",
    control_key="is_control",
    perturbations=("treatment2",),
    perturbations_in_obsm=("treatment2",),
    perturbation_reps={"treatment2":("treatment2_features",),},
    load_target_covariates=True,
    target_covariates={"target0":"label"}
)

In [14]:
train_adata.obs.target0

0      tgt2
1      tgt2
2      tgt0
3      tgt1
4      tgt5
       ... 
844    tgt2
845    tgt5
846    tgt6
847    tgt4
848    tgt3
Name: target0, Length: 700, dtype: object

In [15]:
data = dm.get_data(ood_adata)
data.perturbation_data

{'feats_treatment2_treatment2_features': array([[-0.2398367 , -0.41086947,  0.48962308, ..., -1.3961019 ,
          0.37729195,  1.22658932],
        [-0.51776245, -0.24206427,  0.5885159 , ..., -0.78426819,
         -0.0912344 , -1.19869308],
        [ 0.89552146,  1.54067872, -0.80162358, ...,  1.05384628,
         -0.086736  ,  1.22821388],
        ...,
        [-1.17850547,  0.49433077, -1.87022066, ..., -1.05881183,
          0.668005  , -0.65833829],
        [ 1.91384994, -1.41425024,  1.32707101, ..., -1.06543207,
          1.35049015, -0.85748166],
        [ 0.54126812,  0.55864095, -0.77294967, ...,  1.32923415,
          0.30111007,  0.18055271]], shape=(150, 100))}

In [16]:
dl = ValidationDataLoader(
    {"test0": data},
    IndependentCoupling(),
    batch_size=128,
    # None
)

retrieving indices


In [17]:
batch_dict = dl.sample()
batch_dict.keys()

dict_keys(['test0_treatment2'])

In [18]:
batch_dict["test0_drug4_drug1"].keys()

KeyError: 'test0_drug4_drug1'

In [10]:
batch_dict["test0_drug4_drug1"]["target"].shape

torch.Size([128, 200])

In [11]:
batch_dict["test0_drug4_drug1"]["source"].shape


torch.Size([128, 200])

In [12]:
for k, v in batch_dict["test0_drug4_drug1"]["condition"].items():
    print(k, v.shape)


repr_treatment0_treatment0_label torch.Size([128, 5])
repr_treatment1_treatment1_label torch.Size([128, 5])


In [13]:
data.seen_combinations

[('control', 'control'),
 ('drug5', 'drug5'),
 ('drug2', 'drug2'),
 ('drug2', 'drug5'),
 ('drug1', 'drug1'),
 ('drug1', 'drug3'),
 ('drug5', 'drug1'),
 ('drug1', 'drug2'),
 ('drug4', 'drug2'),
 ('drug5', 'drug3'),
 ('drug3', 'drug3'),
 ('drug3', 'drug2'),
 ('drug4', 'drug1'),
 ('drug2', 'drug3'),
 ('drug5', 'drug2'),
 ('drug2', 'drug4'),
 ('drug1', 'drug5'),
 ('drug3', 'drug4'),
 ('drug2', 'drug1'),
 ('drug1', 'drug4'),
 ('drug3', 'drug1'),
 ('drug5', 'drug4'),
 ('drug4', 'drug3'),
 ('drug4', 'drug5'),
 ('drug4', 'drug4'),
 ('drug3', 'drug5')]